# Weekly Project 5 
## Implementation of global registration 
### Task 1
Today, your task is to implement a global registration algorithm.

It should be able to roughly align two point clouds.
Implement the global registration, and then try the following:

1. Can you fit `r1.pcd` and `r2.pcd`?
2. Can you fit `car1.ply` and `car2.ply`?
The corresponding files are in the `global_registration` folder.


### Task 2 (Challange)
Challanges attempt either or both:
- Implement local registration.

- Attempt to reconstruct the car from the images in `car_challange` folder.

You can use the notebooks from Monday as a starting point.

In [8]:
import open3d as o3d
import matplotlib.pyplot as plt
import numpy as np
import copy
        
# Helper function to draw registrations (reccomended)
def draw_registrations(source, target, transformation = None, recolor = False):
        source_temp = copy.deepcopy(source)
        target_temp = copy.deepcopy(target)
        if(recolor):
            source_temp.paint_uniform_color([1, 0.706, 0])
            target_temp.paint_uniform_color([0, 0.651, 0.929])
        if(transformation is not None):
            source_temp.transform(transformation)
        o3d.visualization.draw_geometries([source_temp, target_temp])

### Office

In [20]:
source = o3d.io.read_point_cloud("global_registration/r1.pcd")
target = o3d.io.read_point_cloud("global_registration/r2.pcd")

# Used for downsampling.
voxel_size = 0.05

# Show models side by side
draw_registrations(source, target)

source_down = source.voxel_down_sample(voxel_size = voxel_size)
target_down = target.voxel_down_sample(voxel_size = voxel_size)

source_down.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius = voxel_size*2, max_nn = 30))
target_down.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius = voxel_size*2, max_nn = 30))

draw_registrations(source_down, target_down)

source_fpfh = o3d.pipelines.registration.compute_fpfh_feature(source_down, o3d.geometry.KDTreeSearchParamHybrid(radius = voxel_size*5, max_nn = 100))
target_fpfh = o3d.pipelines.registration.compute_fpfh_feature(target_down, o3d.geometry.KDTreeSearchParamHybrid(radius = voxel_size*5, max_nn = 100))

In [21]:
# RANSAC

corresp = o3d.pipelines.registration.correspondences_from_features(source_fpfh, target_fpfh)

estimation_method = o3d.pipelines.registration.TransformationEstimationPointToPoint()

transformation = estimation_method.compute_transformation(source_down, target_down, corresp)

distance_threshold = voxel_size*1.5
ransac_result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(source_down, target_down, source_fpfh, target_fpfh, True, distance_threshold, estimation_method)
print(ransac_result)

draw_registrations(source, target, ransac_result.transformation, True)

RegistrationResult with fitness=6.712185e-01, inlier_rmse=3.092435e-02, and correspondence_set size of 3195
Access transformation to get result.


In [22]:
# ICP after RANSAC
source_down = source_down.transform(ransac_result.transformation)
threshold = 0.02
trans_init = np.asarray([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]])

point_to_plane =  o3d.pipelines.registration.TransformationEstimationPointToPlane()

icp_result = o3d.pipelines.registration.registration_icp(
    source_down, target_down, threshold, trans_init,
    point_to_plane)

print(icp_result)
draw_registrations(source_down, target_down, icp_result.transformation, True)

RegistrationResult with fitness=3.626050e-01, inlier_rmse=1.423495e-02, and correspondence_set size of 1726
Access transformation to get result.


### Car


In [25]:
source = o3d.io.read_point_cloud("global_registration/car1.ply")
target = o3d.io.read_point_cloud("global_registration/car2.ply")

# Used for downsampling.
voxel_size = 0.05

# Show models side by side
draw_registrations(source, target)

source_down = source.voxel_down_sample(voxel_size = voxel_size)
target_down = target.voxel_down_sample(voxel_size = voxel_size)

source_down.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius = voxel_size*2, max_nn = 30))
target_down.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius = voxel_size*2, max_nn = 30))

draw_registrations(source_down, target_down)

source_fpfh = o3d.pipelines.registration.compute_fpfh_feature(source_down, o3d.geometry.KDTreeSearchParamHybrid(radius = voxel_size*5, max_nn = 100))
target_fpfh = o3d.pipelines.registration.compute_fpfh_feature(target_down, o3d.geometry.KDTreeSearchParamHybrid(radius = voxel_size*5, max_nn = 100))

In [26]:
# RANSAC

corresp = o3d.pipelines.registration.correspondences_from_features(source_fpfh, target_fpfh)

estimation_method = o3d.pipelines.registration.TransformationEstimationPointToPoint()

transformation = estimation_method.compute_transformation(source_down, target_down, corresp)

distance_threshold = voxel_size*1.5
ransac_result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(source_down, target_down, source_fpfh, target_fpfh, True, distance_threshold, estimation_method)
print(ransac_result)

draw_registrations(source, target, ransac_result.transformation, True)

RegistrationResult with fitness=9.962751e-01, inlier_rmse=3.140243e-02, and correspondence_set size of 3477
Access transformation to get result.


In [27]:
# ICP after RANSAC
source_down = source_down.transform(ransac_result.transformation)
threshold = 0.02
trans_init = np.asarray([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]])

point_to_plane =  o3d.pipelines.registration.TransformationEstimationPointToPlane()

icp_result = o3d.pipelines.registration.registration_icp(
    source_down, target_down, threshold, trans_init,
    point_to_plane)

print(icp_result)
draw_registrations(source_down, target_down, icp_result.transformation, True)

RegistrationResult with fitness=8.091691e-01, inlier_rmse=1.214838e-02, and correspondence_set size of 2824
Access transformation to get result.


### Challenge

In [ ]:
color_raw0 = o3d.io.read_image("car_challange/rgb/0000001.jpg")
depth_raw0 = o3d.io.read_image("car_challange/depth/0000001.png")

rgbd_image0 = o3d.geometry.RGBDImage.create_from_color_and_depth(
    color_raw0, 
    depth_raw0, 
    convert_rgb_to_intensity = True)

camera = o3d.camera.PinholeCameraIntrinsic(
        o3d.camera.PinholeCameraIntrinsicParameters.PrimeSenseDefault)

source = o3d.geometry.PointCloud.create_from_rgbd_image(
    rgbd_image0, camera)
source.transform([[1, 0, 0, 0], [0, -1, 0, 0], [0, 0, -1, 0], [0, 0, 0, 1]])

# Parameters
threshold = 0.02
trans_init = np.asarray([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]])
voxel_size = 0.05

source = source.voxel_down_sample(voxel_size=0.05)

# 1722 total images
# starting img 0000001
# last image   0001722

for i in range(2, 1723):
    
    br = ''
    if i < 1000:
        br = '0'
        if i < 100:
            br = '00'
            if i<10:
                br = '000'
    
    br = br + str(i)
    
    if int(br) % 100 == 0:
        print('Image num:' + br)

    if np.max(np.asarray(source.points).shape) > 100000:    # downsample base if it gets too big
        source = source.voxel_down_sample(voxel_size=0.02)
        
    color_raw1 = o3d.io.read_image("car_challange/rgb/000" + br + ".jpg")
    depth_raw1 = o3d.io.read_image("car_challange/depth/000" + br + ".png")
    rgbd_image1 = o3d.geometry.RGBDImage.create_from_color_and_depth(
        color_raw1, 
        depth_raw1, 
        convert_rgb_to_intensity = True)
    target = o3d.geometry.PointCloud.create_from_rgbd_image(
        rgbd_image1, camera)
    target.transform([[1, 0, 0, 0], [0, -1, 0, 0], [0, 0, -1, 0], [0, 0, 0, 1]])

    target = target.voxel_down_sample(voxel_size=0.05)
    
    source.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius = 0.5, max_nn = 30))
    target.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius = 0.5, max_nn = 30))

    point_to_plane =  o3d.pipelines.registration.TransformationEstimationPointToPlane()
    icp_result = o3d.pipelines.registration.registration_icp(
        source, target, threshold, trans_init,
        point_to_plane)
    source.transform(icp_result.transformation)
    new = source + target

    
    # new_down = new.voxel_down_sample(voxel_size=0.05)

    source = copy.deepcopy(new)
    
    if int(br) % 1000 == 0:
        break
draw_registrations(source, target, icp_result.transformation, True)
draw_registrations(source, target, icp_result.transformation)

Image num:0100
Image num:0200
Image num:0300
Image num:0400


In [68]:
source = o3d.geometry.PointCloud.create_from_rgbd_image(
    rgbd_image0, camera)
source = source.voxel_down_sample(voxel_size=0.01)
print(np.max(np.asarray(source.points).shape))

51197


In [72]:
draw_registrations(source, target, icp_result.transformation, True)
draw_registrations(source, target, icp_result.transformation)